In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


#don't use

In [ ]:
# 📦 Install dependencies
!pip install -q scikit-learn lightgbm xgboost imbalanced-learn joblib

# 📌 Imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from collections import Counter
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# 📂 Load dataset from Drive
df = pd.read_csv('/content/drive/MyDrive/final_augmented_only_train_data.csv')

# 🎯 Extract features and labels
X = df.drop(columns=['file_key', 'final_emotion'])
y = df['final_emotion']

# ✂️ Stratified split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)

# 🧼 Scale
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 💉 SMOTE
print("Before SMOTE:", Counter(y_train))
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)
print("After SMOTE:", Counter(y_train_sm))

# 🔍 LGBM Feature Selection on audio only
feature_names = X.columns
text_cols = [col for col in X.columns if col.startswith('text_')]
audio_cols = [col for col in X.columns if col not in text_cols]

X_audio = pd.DataFrame(X_train_sm, columns=feature_names)[audio_cols]
lgbm_fs = LGBMClassifier(random_state=42)
lgbm_fs.fit(X_audio, y_train_sm)
importances = pd.Series(lgbm_fs.feature_importances_, index=audio_cols)
top_audio = importances.sort_values(ascending=False).head(150).index.tolist()

# ✅ Final feature list
selected_features = top_audio + text_cols

# 📌 Final train/val sets
X_train_final = pd.DataFrame(X_train_sm, columns=feature_names)[selected_features]
X_val_final = pd.DataFrame(X_val_scaled, columns=feature_names)[selected_features]


# 📊 Define models
models = {
    'LGBM': LGBMClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    'SVM': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

from sklearn.preprocessing import LabelEncoder

# 🎯 Encode labels to integers for models that require numeric classes (like XGBoost)
label_encoder = LabelEncoder()
y_train_sm_encoded = label_encoder.fit_transform(y_train_sm)
y_val_encoded = label_encoder.transform(y_val)
class_names = label_encoder.classes_


# 🔁 Train and evaluate each model
for name, model in models.items():
    print(f"\n🔍 Training and evaluating: {name}")

    # Use encoded labels only for XGBoost
    if name == 'XGBoost':
        model.fit(X_train_final, y_train_sm_encoded)
        proba = model.predict_proba(X_val_final)
        classes = class_names
        y_val_current = y_val_encoded
    else:
        model.fit(X_train_final, y_train_sm)
        proba = model.predict_proba(X_val_final)
        classes = model.classes_
        y_val_current = y_val

    # 🎯 Threshold tuning
    best_thresholds = {}
    y_val_bin = pd.get_dummies(y_val_current, columns=classes)
    final_preds = []

    for i, cls in enumerate(classes):
        cls_scores = []
        thresholds = np.arange(0.1, 0.9, 0.01)
        y_true = y_val_bin.iloc[:, i]

        for t in thresholds:
            y_pred = (proba[:, i] >= t).astype(int)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            cls_scores.append((t, f1))

        best_t = max(cls_scores, key=lambda x: x[1])[0]
        best_thresholds[cls] = best_t

    # 🧠 Final predictions using best thresholds
    for row in proba:
        row_pred = None
        for i, cls in enumerate(classes):
            if row[i] >= best_thresholds[cls]:
                row_pred = cls
                break
        if row_pred:
            final_preds.append(row_pred)
        else:
            final_preds.append(classes[np.argmax(row)])

    # Decode if predictions are numeric (only for XGBoost)
    if name == 'XGBoost':
        final_preds = label_encoder.inverse_transform([class_names.tolist().index(p) for p in final_preds])
        y_val_eval = y_val
    else:
        y_val_eval = y_val

    # 📊 Evaluation
    print(f"\n✅ Classification Report for {name}:\n")
    print(classification_report(y_val_eval, final_preds))
    print(f"\n🧩 Confusion Matrix for {name}:\n")
    print(confusion_matrix(y_val_eval, final_preds))



Before SMOTE: Counter({'fear': 295, 'happy': 295, 'sad': 295, 'angry': 295, 'neutral': 294})
After SMOTE: Counter({'fear': 295, 'happy': 295, 'sad': 295, 'angry': 295, 'neutral': 295})
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001909 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25500
[LightGBM] [Info] Number of data points in the train set: 1475, number of used features: 100
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

#FINAL MODEL FOR TRAINING

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1) Install dependencies
# ─────────────────────────────────────────────────────────────────────────────
!pip install -q scikit-learn lightgbm xgboost imbalanced-learn joblib

# ─────────────────────────────────────────────────────────────────────────────
# 2) Imports
# ─────────────────────────────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import joblib
from collections import Counter

from sklearn.model_selection      import train_test_split
from sklearn.preprocessing       import RobustScaler
from sklearn.ensemble            import VotingClassifier, RandomForestClassifier
from lightgbm                    import LGBMClassifier
from xgboost                     import XGBClassifier
from imblearn.over_sampling      import SMOTE
from sklearn.metrics             import classification_report, confusion_matrix

# ─────────────────────────────────────────────────────────────────────────────
# 3) Load augmented‐train CSV
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv('/content/drive/MyDrive/final_augmented_only_train_data.csv')

# Features & labels
X = df.drop(columns=['file_key','final_emotion'])
y = df['final_emotion']

# ─────────────────────────────────────────────────────────────────────────────
# 4) Stratified split (85/15)
# ─────────────────────────────────────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)

# ─────────────────────────────────────────────────────────────────────────────
# 5) Scale
# ─────────────────────────────────────────────────────────────────────────────
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

# ─────────────────────────────────────────────────────────────────────────────
# 6) SMOTE on train
# ─────────────────────────────────────────────────────────────────────────────
print("Before SMOTE:", Counter(y_train))
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)
print("After SMOTE:", Counter(y_train_sm))

# ─────────────────────────────────────────────────────────────────────────────
# 7) Audio vs. Text feature split & LGBM FS
# ─────────────────────────────────────────────────────────────────────────────
feature_names = X.columns
text_cols     = [c for c in feature_names if c.startswith('text_')]
audio_cols    = [c for c in feature_names if c not in text_cols]

# fit LGBM on audio only
X_train_sm_df = pd.DataFrame(X_train_sm, columns=feature_names)
lgbm_fs       = LGBMClassifier(random_state=42)
lgbm_fs.fit(X_train_sm_df[audio_cols], y_train_sm)

# pick top 150 audio features
importances   = pd.Series(lgbm_fs.feature_importances_, index=audio_cols)
top_audio     = importances.nlargest(150).index.tolist()

# final feature list = top_audio + all text
selected_features = top_audio + text_cols

# build final train & val sets
X_train_final = X_train_sm_df[selected_features]
X_val_final   = pd.DataFrame(X_val_scaled, columns=feature_names)[selected_features]

# ─────────────────────────────────────────────────────────────────────────────
# 8) Train VotingClassifier (soft voting)
# ─────────────────────────────────────────────────────────────────────────────
lgbm = LGBMClassifier(random_state=42)
xgb  = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
rf   = RandomForestClassifier(class_weight='balanced', random_state=42)

voting = VotingClassifier(
    estimators=[('lgbm', lgbm), ('xgb', xgb), ('rf', rf)],
    voting='soft',
    n_jobs=-1
)
voting.fit(X_train_final, y_train_sm)

# ─────────────────────────────────────────────────────────────────────────────
# 9) Predict & Evaluate (argmax over soft‑voting)
# ─────────────────────────────────────────────────────────────────────────────
y_pred = voting.predict(X_val_final)

print("\n✅ Classification Report:\n")
print(classification_report(y_val, y_pred))
print("\n🧩 Confusion Matrix:\n")
print(confusion_matrix(y_val, y_pred))

# ─────────────────────────────────────────────────────────────────────────────
# 10) Save model & preprocessing
# ─────────────────────────────────────────────────────────────────────────────
out_dir = "/content/drive/MyDrive/final_model_assets"
os.makedirs(out_dir, exist_ok=True)

joblib.dump(voting,           f"{out_dir}/voting_model.pkl")
joblib.dump(scaler,           f"{out_dir}/scaler.pkl")
joblib.dump(selected_features,f"{out_dir}/features_used.pkl")

print(f"\n📁 Artifacts saved to {out_dir}")


Before SMOTE: Counter({'fear': 295, 'happy': 295, 'sad': 295, 'angry': 295, 'neutral': 294})
After SMOTE: Counter({'fear': 295, 'happy': 295, 'sad': 295, 'angry': 295, 'neutral': 295})
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001555 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25500
[LightGBM] [Info] Number of data points in the train set: 1475, number of used features: 100
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

#TESTING

In [ ]:
import joblib
import numpy as np
import pandas as pd

# Load saved assets
model = joblib.load("/content/drive/MyDrive/final_model_assets/voting_model.pkl")
scaler = joblib.load("/content/drive/MyDrive/final_model_assets/scaler.pkl")
selected_features = joblib.load("/content/drive/MyDrive/final_model_assets/features_used.pkl")
thresholds = joblib.load("/content/drive/MyDrive/final_model_assets/thresholds.pkl")

def predict_emotion(audio_path, raw_text):
    # Extract audio features
    audio_feats = extract_audio_features(audio_path)  # returns dict
    text_feats = extract_text_features_from_raw(raw_text)  # returns dict

    # Combine and force correct ordering
    combined = {**audio_feats, **text_feats}
    row = [combined.get(feat, 0) for feat in selected_features]  # order and fill
    input_df = pd.DataFrame([row], columns=selected_features)

    # Scale
    input_scaled = scaler.transform(input_df)

    # Predict probabilities
    probs = model.predict_proba(input_scaled)[0]
    labels = model.classes_

    # Threshold logic
    for i, label in enumerate(labels):
        if probs[i] >= thresholds[label]:
            print(f"\n🎯 Emotion: **{label}** (Threshold hit)")
            break
    else:
        fallback = labels[np.argmax(probs)]
        print(f"\n⚠️ Fallback Emotion: **{fallback}** (Max Prob)")

    print("\n🔢 Class Probabilities:")
    for i, label in enumerate(labels):
        print(f"  {label}: {probs[i]:.3f} (≥ {thresholds[label]:.3f})")                                              # ─────────────────────────────────────────────────────────────────────────────
# 1) Install everything you need
# ─────────────────────────────────────────────────────────────────────────────
!pip install -q librosa transformers sentence-transformers joblib scikit-learn                        # ─────────────────────────────────────────────────────────────────────────────
# 2) (Colab only) Mount Drive so your .pkl files are visible
# ─────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/final_model_assets"
except ImportError:
    BASE = "./final_model_assets"   # adjust if running locally                                                       # ─────────────────────────────────────────────────────────────────────────────
# 3) Imports & Load Artifacts
# ─────────────────────────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import joblib
import librosa
import torch
from transformers import AutoTokenizer, AutoModel
from google.colab import files  # for the final upload

# load your model + preprocessing objects
voting_model      = joblib.load(os.path.join(BASE, "voting_model.pkl"))
scaler            = joblib.load(os.path.join(BASE, "scaler.pkl"))
thresholds        = joblib.load(os.path.join(BASE, "thresholds.pkl"))

# if you have a separate features_used.pkl, you can ignore it now—
# we'll derive our column order from scaler.feature_names_in_
if not hasattr(scaler, "feature_names_in_"):
    raise ValueError("Your scaler must have been fit on a DataFrame; it needs feature_names_in_")
FEATURE_ORDER = list(scaler.feature_names_in_)

# ensure thresholds is a simple dict[label, float]
if not isinstance(thresholds, dict):
    thresholds = dict(thresholds)                                                                                        # ─────────────────────────────────────────────────────────────────────────────
# 4) Text Embedding Helpers (with max_length to suppress warnings)
# ─────────────────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
text_model= AutoModel.from_pretrained("ai4bharat/indic-bert")
MAX_LEN   = 128

def get_mean_pooling_embedding(text: str) -> np.ndarray:
    toks = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
    with torch.no_grad():
        out = text_model(**toks).last_hidden_state
    mask = toks["attention_mask"].unsqueeze(-1).expand(out.size()).float()
    summed      = torch.sum(out * mask, dim=1)
    summed_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
    return (summed / summed_mask).squeeze().cpu().numpy()

def extract_text_features(text: str) -> dict:
    emb = get_mean_pooling_embedding(text)
    return {f"text_{i}": float(emb[i]) for i in range(len(emb))}                                                     # ─────────────────────────────────────────────────────────────────────────────
# 5) Audio Feature Extraction
# ─────────────────────────────────────────────────────────────────────────────
def extract_audio_features(path: str) -> dict:
    y, sr = librosa.load(path, sr=None)
    feats = {}
    # MFCC (13)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        feats[f"mfcc_{i}"] = float(np.mean(mfcc[i]))
    # ZCR & RMS
    feats["zcr"] = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    feats["rms"] = float(np.mean(librosa.feature.rms(y=y)))
    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(chroma.shape[0]):
        feats[f"chroma_{i}"] = float(np.mean(chroma[i]))
    # Spectral Contrast
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        feats[f"contrast_{i}"] = float(np.mean(contrast[i]))
    # Tonnetz
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    for i in range(tonnetz.shape[0]):
        feats[f"tonnetz_{i}"] = float(np.mean(tonnetz[i]))
    # Mel Spectrogram (up to 128)
    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    for i in range(min(128, mel.shape[0])):
        feats[f"mel_{i}"] = float(np.mean(mel[i]))
    return feats                                                                                                                        # ─────────────────────────────────────────────────────────────────────────────
# 6) Prediction Function (uses FEATURE_ORDER for exact match)
# ─────────────────────────────────────────────────────────────────────────────
def predict_emotion(audio_path: str, raw_text: str):
    # 1) extract both modalities
    a_feats = extract_audio_features(audio_path)
    t_feats = extract_text_features(raw_text)
    combined = {**a_feats, **t_feats}

    # 2) build DataFrame with the scaler’s exact columns
    row = [combined.get(f, 0.0) for f in FEATURE_ORDER]
    df  = pd.DataFrame([row], columns=FEATURE_ORDER)

    # 3) scale & predict
    Xs    = scaler.transform(df)
    probs = voting_model.predict_proba(Xs)[0]
    labels= voting_model.classes_

    # 4) threshold logic (pick highest margin over threshold)
    # 4) threshold logic (pick the class with the largest margin over its threshold)
    candidates = []
    for prob, label in zip(probs, labels):
        thr    = thresholds.get(label, 0.5)
        margin = prob - thr
        if margin >= 0:
            candidates.append((label, margin, prob, thr))

    if candidates:
        # choose the class that exceeded its threshold by the biggest amount
        best_label, best_margin, best_prob, best_thr = max(candidates, key=lambda x: x[1])
        print(f"🎯 Emotion: **{best_label}** "
              f"(prob={best_prob:.3f} ≥ thr={best_thr:.3f}, margin={best_margin:.3f})")
    else:
        # no threshold hit → fallback to highest raw probability
        fb = labels[np.argmax(probs)]
        print(f"⚠️ Fallback Emotion: **{fb}** (max_prob={max(probs):.3f})")


    # 5) print all
    print("\n🔢 Probabilities:")
    for prob, label in zip(probs, labels):
        thr_part = f" (thr={thresholds[label]:.3f})" if label in thresholds else ""
        print(f"  {label}: {prob:.3f}{thr_part}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

In [ ]:
# 7) Upload & Inference
# ─────────────────────────────────────────────────────────────────────────────
uploaded = files.upload()
audio_file = next(iter(uploaded.keys()))
text_input = input("Enter transcript/text: ")
predict_emotion(audio_file, text_input)

Saving 434a16c6-f786-405e-83f2-74c602e560b6.wav to 434a16c6-f786-405e-83f2-74c602e560b6.wav
Enter transcript/text: കാട്ടിലെ തടി തേവരുടെ ആന വലിയെടാ വലി.
🎯 Emotion: **happy** (prob=0.694 ≥ thr=0.260, margin=0.434)

🔢 Probabilities:
  angry: 0.056 (thr=0.100)
  fear: 0.093 (thr=0.480)
  happy: 0.694 (thr=0.260)
  neutral: 0.092 (thr=0.540)
  sad: 0.065 (thr=0.600)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


#GENERATES TRANSCRIPTS AND PREDICT EMOTION

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1) Install & imports
# ─────────────────────────────────────────────────────────────────────────────
!pip install -q librosa transformers joblib scikit-learn

import os
import joblib
import librosa
import torch
import numpy as np
import pandas as pd

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModel
)
from google.colab import files

# ─────────────────────────────────────────────────────────────────────────────
# 2) Load trained artifacts
# ─────────────────────────────────────────────────────────────────────────────
BASE = "/content/drive/MyDrive/final_model_assets"
voting_model  = joblib.load(os.path.join(BASE, "voting_model.pkl"))
scaler        = joblib.load(os.path.join(BASE, "scaler.pkl"))
FEATURE_ORDER = joblib.load(os.path.join(BASE, "features_used.pkl"))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ─────────────────────────────────────────────────────────────────────────────
# 3) Load Whisper model & processor
# ─────────────────────────────────────────────────────────────────────────────
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model_whisper = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(DEVICE)

# Set the task and language for forced decoder
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ml", task="transcribe"
)

def asr_whisper_ml(path: str) -> str:
    # load audio
    speech, sr = librosa.load(path, sr=16000)
    inputs = processor(speech, sampling_rate=sr, return_tensors="pt").to(DEVICE)

    # generate transcription without specifying max_new_tokens
    gen = model_whisper.generate(
        **inputs,
        forced_decoder_ids=forced_decoder_ids,
        # <--- remove max_new_tokens entirely
    )

    return processor.batch_decode(gen, skip_special_tokens=True)[0]

# ─────────────────────────────────────────────────────────────────────────────
# 4) Feature‑extraction helpers (same as before)
# ─────────────────────────────────────────────────────────────────────────────
tokenizer_txt = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
text_model    = AutoModel.from_pretrained("ai4bharat/indic-bert")
MAX_LEN       = 128

def extract_text_features(text: str) -> dict:
    toks = tokenizer_txt(text, return_tensors="pt",
                         truncation=True, padding="max_length",
                         max_length=MAX_LEN)
    with torch.no_grad():
        out = text_model(**toks).last_hidden_state
    mask   = toks["attention_mask"].unsqueeze(-1).expand(out.size()).float()
    summed = torch.sum(out * mask, dim=1)
    denom  = torch.clamp(mask.sum(dim=1), min=1e-9)
    emb    = (summed/denom).squeeze().cpu().numpy()
    TEXT_DIMS = len([f for f in FEATURE_ORDER if f.startswith("text_")])
    emb = emb[:TEXT_DIMS]
    return {f"text_{i}": float(emb[i]) for i in range(TEXT_DIMS)}

def extract_audio_features(path: str) -> dict:
    y, sr = librosa.load(path, sr=None)
    feats = {}
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        feats[f"mfcc_{i}"] = float(mfcc[i].mean())
    feats["zcr"] = float(librosa.feature.zero_crossing_rate(y).mean())
    feats["rms"] = float(librosa.feature.rms(y=y).mean())
    chroma   = librosa.feature.chroma_stft(y=y, sr=sr)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    tonnetz  = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    mel      = librosa.feature.melspectrogram(y=y, sr=sr)
    for arr, pref in [(chroma,"chroma"),
                      (contrast,"contrast"),
                      (tonnetz,"tonnetz")]:
        for j in range(arr.shape[0]):
            feats[f"{pref}_{j}"] = float(arr[j].mean())
    for j in range(min(128, mel.shape[0])):
        feats[f"mel_{j}"] = float(mel[j].mean())
    return feats

# ─────────────────────────────────────────────────────────────────────────────
# 5) Unified inference function
# ─────────────────────────────────────────────────────────────────────────────
# pip install the transliteration helper
# ─────────────────────────────────────────────────────────────────────────────
!pip install -q indic-transliteration

from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def transcribe_and_predict(audio_path: str, llm_response: bool=False):
    # 1) ASR → Devanagari transcript
    dev_text = asr_whisper_ml(audio_path)
    # 2) Transliterate to Malayalam script
    transcript = transliterate(dev_text, sanscript.DEVANAGARI, sanscript.MALAYALAM)
    print(f"\n📝 Malayalam Transcript:\n{transcript}")

    # -- rest stays the same: feature extraction, predict emotion, optional LLM --
    ...


    # 2) Extract features
    a_feats   = extract_audio_features(audio_path)
    t_feats   = extract_text_features(transcript)
    combined  = {**a_feats, **t_feats}

    # 3) Build input vector
    row       = [combined.get(f, 0.0) for f in FEATURE_ORDER]
    X_input   = np.array(row).reshape(1, -1)

    # 4) Scale & predict
    Xs        = scaler.transform(X_input)
    emotion   = voting_model.predict(Xs)[0]
    probs     = voting_model.predict_proba(Xs)[0]

    print(f"\n🎯 Detected Emotion: {emotion}")
    print("\n🔢 Probabilities:")
    for lbl, p in zip(voting_model.classes_, probs):
        print(f"  {lbl}: {p:.3f}")

    # 5) (Optional) LLM reply in Malayalam
    if llm_response:
        # Insert your LLM call here (same as before)
        pass

# ─────────────────────────────────────────────────────────────────────────────
# 6) Upload & run
# ─────────────────────────────────────────────────────────────────────────────
uploaded    = files.upload()
audio_file  = next(iter(uploaded.keys()))
transcribe_and_predict(audio_file, llm_response=False)


Saving 59e67920-49cf-46ea-835e-9b5b9196f248.wav to 59e67920-49cf-46ea-835e-9b5b9196f248 (3).wav

📝 Malayalam Transcript:
 Aayu ka matthi mahabalam

🎯 Detected Emotion: neutral

🔢 Probabilities:
  angry: 0.006
  fear: 0.073
  happy: 0.102
  neutral: 0.765
  sad: 0.054
